# 🛍️ E-Commerce Purchase Prediction
**Machine Learning untuk Prediksi Pembelian pada Platform E-Commerce**

Notebook ini membangun model ML untuk memprediksi apakah seorang user akan melakukan pembelian berdasarkan perilaku browsing mereka.

**Target**: `ordered` — 0 (tidak beli) / 1 (beli)

---

## 1. 📦 Install & Import Library

In [ ]:
!pip install xgboost lightgbm imbalanced-learn shap plotly -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
import shap
import joblib
import json
import os

print('✅ Semua library berhasil diimport!')


## 2. 📂 Load Dataset

In [ ]:
from google.colab import files
print('📤 Upload file training_sample.csv Anda:')
uploaded = files.upload()


In [ ]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f'✅ Dataset berhasil dimuat!')
print(f'📊 Jumlah baris   : {df.shape[0]:,}')
print(f'📊 Jumlah kolom   : {df.shape[1]}')
df.head()


## 3. 🔍 Exploratory Data Analysis (EDA)

In [ ]:
print('=== INFO DATASET ===')
df.info()


In [ ]:
# Distribusi target (class imbalance)
target_counts = df['ordered'].value_counts()
target_pct = df['ordered'].value_counts(normalize=True) * 100

print('=== DISTRIBUSI TARGET ===')
for cls, cnt in target_counts.items():
    label = 'Beli (1)' if cls == 1 else 'Tidak Beli (0)'
    print(f'  {label}: {cnt:,} ({target_pct[cls]:.2f}%)')

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'domain'}, {'type': 'bar'}]])
fig.add_trace(go.Pie(
    labels=['Tidak Beli (0)', 'Beli (1)'],
    values=target_counts.values, hole=0.4,
    marker_colors=['#EF553B', '#00CC96']
), row=1, col=1)
fig.add_trace(go.Bar(
    x=['Tidak Beli (0)', 'Beli (1)'],
    y=target_counts.values,
    marker_color=['#EF553B', '#00CC96'],
    text=[f'{v:,}' for v in target_counts.values],
    textposition='auto'
), row=1, col=2)
fig.update_layout(title_text='Distribusi Target: Ordered', height=400)
fig.show()


In [ ]:
# Correlation heatmap
feature_cols = [c for c in df.columns if c not in ['UserID', 'ordered']]
plt.figure(figsize=(14, 10))
corr = df[feature_cols + ['ordered']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdYlGn', center=0, linewidths=0.5)
plt.title('Correlation Matrix Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Korelasi fitur dengan target
corr_target = corr['ordered'].drop('ordered').sort_values(ascending=False)
print('\n=== KORELASI FITUR DENGAN TARGET ===')
print(corr_target.round(4))


In [ ]:
# Rata-rata fitur per kelas
feature_means = df.groupby('ordered')[feature_cols].mean().T
feature_means.columns = ['Tidak Beli', 'Beli']
feature_means['Ratio'] = (feature_means['Beli'] / (feature_means['Tidak Beli'] + 1e-9)).round(2)
feature_means = feature_means.sort_values('Ratio', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='Tidak Beli (0)', x=feature_means.index, y=feature_means['Tidak Beli'], marker_color='#EF553B'))
fig.add_trace(go.Bar(name='Beli (1)', x=feature_means.index, y=feature_means['Beli'], marker_color='#00CC96'))
fig.update_layout(title='Rata-rata Fitur per Kelas Target', barmode='group', xaxis_tickangle=-45, height=500)
fig.show()


## 4. 🔧 Preprocessing & Feature Engineering

In [ ]:
X = df[feature_cols].copy()
y = df['ordered'].copy()

# Feature Engineering
X['total_activity']       = X[feature_cols].sum(axis=1)
X['basket_intent']        = X['basket_icon_click'] + X['basket_add_list'] + X['basket_add_detail']
X['checkout_intent']      = X['saw_checkout'] + X['closed_minibasket_click']
X['product_info_check']   = X['checked_delivery_detail'] + X['checked_returns_detail'] + X['saw_sizecharts']
X['engagement_score']     = X['promo_banner_click'] + X['image_picker'] + X['detail_wishlist_add']

print('✅ Feature engineering selesai!')
print(f'Total fitur: {X.shape[1]}')
print(f'Fitur baru: total_activity, basket_intent, checkout_intent, product_info_check, engagement_score')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'\nTraining: {X_train.shape[0]:,} baris | Test: {X_test.shape[0]:,} baris')


## 5. 🤖 Training Model (Multi-Model Comparison)

In [ ]:
scale_pw = (y_train==0).sum() / (y_train==1).sum()

models = {
    'Logistic Regression': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
    ]),
    'Random Forest': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=10,
                                        class_weight='balanced', random_state=42, n_jobs=-1))
    ]),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_pw, random_state=42,
        eval_metric='logloss', verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        class_weight='balanced', random_state=42, verbose=-1
    )
}

results = {}
for name, model in models.items():
    print(f'🔄 Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc  = average_precision_score(y_test, y_prob)
    f1      = f1_score(y_test, y_pred)
    prec    = precision_score(y_test, y_pred)
    rec     = recall_score(y_test, y_pred)
    results[name] = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
                     'ROC-AUC': roc_auc, 'PR-AUC': pr_auc,
                     'F1': f1, 'Precision': prec, 'Recall': rec}
    print(f'   ✅ ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f} | F1: {f1:.4f}')

print('\n🎉 Semua model selesai dilatih!')


## 6. 📊 Evaluasi & Perbandingan Model

In [ ]:
comparison = pd.DataFrame({
    name: {'ROC-AUC': res['ROC-AUC'], 'PR-AUC': res['PR-AUC'],
           'F1-Score': res['F1'], 'Precision': res['Precision'], 'Recall': res['Recall']}
    for name, res in results.items()
}).T.round(4).sort_values('ROC-AUC', ascending=False)

print('=== PERBANDINGAN MODEL ===')
print(comparison.to_string())

best_model_name = comparison['ROC-AUC'].idxmax()
print(f'\n🏆 Model terbaik: {best_model_name} (ROC-AUC: {comparison.loc[best_model_name, "ROC-AUC"]:.4f})')


In [ ]:
# ROC Curve
colors = px.colors.qualitative.Set2
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
    line=dict(dash='dash', color='gray'), name='Random Baseline'))
for i, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
        name=f"{name} (AUC={res['ROC-AUC']:.4f})",
        line=dict(color=colors[i], width=2)))
fig.update_layout(title='ROC Curve - Semua Model',
    xaxis_title='False Positive Rate', yaxis_title='True Positive Rate', height=500)
fig.show()


In [ ]:
# Confusion Matrix - Best Model
best_res = results[best_model_name]
cm = confusion_matrix(y_test, best_res['y_pred'])
fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
    x=['Predicted: Tidak Beli', 'Predicted: Beli'],
    y=['Actual: Tidak Beli', 'Actual: Beli'],
    title=f'Confusion Matrix — {best_model_name}', aspect='auto')
fig.show()

print(f'\nClassification Report — {best_model_name}:')
print(classification_report(y_test, best_res['y_pred'], target_names=['Tidak Beli', 'Beli']))


## 7. 🔬 SHAP Feature Importance

In [ ]:
shap_model_name = 'LightGBM' if 'LightGBM' in results else 'XGBoost'
shap_model = results[shap_model_name]['model']
actual_model = shap_model.named_steps['clf'] if hasattr(shap_model, 'named_steps') else shap_model

X_shap = X_test.sample(min(2000, len(X_test)), random_state=42)
explainer = shap.TreeExplainer(actual_model)
shap_values = explainer.shap_values(X_shap)
shap_vals = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_vals, X_shap, plot_type='bar', show=False)
plt.title('SHAP Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_vals, X_shap, show=False)
plt.title('SHAP Beeswarm Plot', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. 🎯 Threshold Optimization

In [ ]:
best_probs = best_res['y_prob']
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores, prec_scores, rec_scores = [], [], []
for t in thresholds:
    y_pred_t = (best_probs >= t).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_t, zero_division=0))
    prec_scores.append(precision_score(y_test, y_pred_t, zero_division=0))
    rec_scores.append(recall_score(y_test, y_pred_t, zero_division=0))

best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

fig = go.Figure()
fig.add_trace(go.Scatter(x=thresholds, y=f1_scores, name='F1-Score', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=thresholds, y=prec_scores, name='Precision', line=dict(color='green')))
fig.add_trace(go.Scatter(x=thresholds, y=rec_scores, name='Recall', line=dict(color='orange')))
fig.add_vline(x=best_threshold, line_dash='dash', line_color='red',
    annotation_text=f'Optimal: {best_threshold:.2f}')
fig.update_layout(title='Threshold Optimization', xaxis_title='Threshold', height=450)
fig.show()

print(f'Threshold optimal: {best_threshold:.2f} | F1 optimal: {best_f1:.4f}')

y_pred_optimal = (best_probs >= best_threshold).astype(int)
print(f'\nClassification Report dengan Threshold Optimal:')
print(classification_report(y_test, y_pred_optimal, target_names=['Tidak Beli', 'Beli']))


## 9. 💾 Save Model

In [ ]:
os.makedirs('model_artifacts', exist_ok=True)

best_model = results[best_model_name]['model']
joblib.dump(best_model, 'model_artifacts/best_model.pkl')

model_metadata = {
    'model_name': best_model_name,
    'optimal_threshold': float(best_threshold),
    'roc_auc': float(best_res['ROC-AUC']),
    'pr_auc': float(best_res['PR-AUC']),
    'f1_score': float(best_f1),
    'features': list(X.columns),
    'original_features': feature_cols,
    'engineered_features': ['total_activity','basket_intent','checkout_intent',
                             'product_info_check','engagement_score']
}
with open('model_artifacts/model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)

print('✅ Model tersimpan!')
print('   model_artifacts/best_model.pkl')
print('   model_artifacts/model_metadata.json')

import shutil
shutil.make_archive('model_artifacts', 'zip', 'model_artifacts')
files.download('model_artifacts.zip')
print('\n📥 model_artifacts.zip berhasil didownload!')


## 10. 🧪 Demo Prediksi

In [ ]:
def predict_purchase(user_dict, model, threshold, all_feature_cols, orig_features):
    df_u = pd.DataFrame([user_dict])
    df_u['total_activity']     = df_u[orig_features].sum(axis=1)
    df_u['basket_intent']      = df_u['basket_icon_click'] + df_u['basket_add_list'] + df_u['basket_add_detail']
    df_u['checkout_intent']    = df_u['saw_checkout'] + df_u['closed_minibasket_click']
    df_u['product_info_check'] = df_u['checked_delivery_detail'] + df_u['checked_returns_detail'] + df_u['saw_sizecharts']
    df_u['engagement_score']   = df_u['promo_banner_click'] + df_u['image_picker'] + df_u['detail_wishlist_add']
    df_u = df_u[all_feature_cols]
    prob = model.predict_proba(df_u)[0][1]
    pred = int(prob >= threshold)
    return {
        'Prediksi': 'AKAN BELI ✅' if pred == 1 else 'TIDAK BELI ❌',
        'Probabilitas': f'{prob*100:.1f}%',
        'Confidence': 'Tinggi' if abs(prob-0.5) > 0.3 else 'Sedang' if abs(prob-0.5) > 0.1 else 'Rendah'
    }

user_beli = {
    'basket_icon_click':1,'basket_add_list':1,'basket_add_detail':1,'sort_by':1,
    'image_picker':1,'account_page_click':0,'promo_banner_click':1,'detail_wishlist_add':1,
    'list_size_dropdown':1,'closed_minibasket_click':1,'checked_delivery_detail':1,
    'checked_returns_detail':0,'sign_in':1,'saw_checkout':1,'saw_sizecharts':1,
    'saw_delivery':1,'saw_account_upgrade':0,'saw_homepage':1,
    'device_mobile':1,'device_computer':0,'device_tablet':0,'returning_user':1,'loc_uk':1
}
user_tidak = {
    'basket_icon_click':0,'basket_add_list':0,'basket_add_detail':0,'sort_by':1,
    'image_picker':0,'account_page_click':0,'promo_banner_click':0,'detail_wishlist_add':0,
    'list_size_dropdown':0,'closed_minibasket_click':0,'checked_delivery_detail':0,
    'checked_returns_detail':0,'sign_in':0,'saw_checkout':0,'saw_sizecharts':0,
    'saw_delivery':0,'saw_account_upgrade':0,'saw_homepage':1,
    'device_mobile':0,'device_computer':1,'device_tablet':0,'returning_user':0,'loc_uk':1
}

print('=== DEMO PREDIKSI ===')
print('\n👤 User A (aktif, banyak interaksi basket & checkout):')
for k, v in predict_purchase(user_beli, best_model, best_threshold, list(X.columns), feature_cols).items():
    print(f'   {k}: {v}')

print('\n👤 User B (pasif, hanya lihat homepage):')
for k, v in predict_purchase(user_tidak, best_model, best_threshold, list(X.columns), feature_cols).items():
    print(f'   {k}: {v}')


## 11. 📋 Ringkasan & Insight Bisnis

In [ ]:
print('='*60)
print('📊 RINGKASAN - E-COMMERCE PURCHASE PREDICTION')
print('='*60)
print(f'''
DATASET:
  Total sessions   : {len(df):,}
  Fitur            : {len(feature_cols)} fitur perilaku + 5 engineered
  Class imbalance  : {df["ordered"].mean()*100:.1f}% user melakukan pembelian

MODEL TERBAIK: {best_model_name}
  ROC-AUC          : {best_res["ROC-AUC"]:.4f}
  PR-AUC           : {best_res["PR-AUC"]:.4f}
  F1-Score optimal : {best_f1:.4f}
  Threshold optimal: {best_threshold:.2f}

INSIGHT BISNIS:
  1. saw_checkout + basket_add = signal beli paling kuat
  2. Kirim notifikasi/diskon ke user yang add to basket tapi belum checkout
  3. Returning user memiliki konversi lebih tinggi — prioritaskan loyalty program
  4. Optimalkan UX checkout di mobile

NEXT STEPS:
  → Deploy ke Streamlit (gunakan model_artifacts.zip)
  → Upload ke GitHub
  → Monitor & retrain model setiap bulan
''')
print('='*60)
